# 14 · Use Case — IoT Sensor Monitoring & Predictive Maintenance

Combine **forecasting** + **anomaly detection** to watch machine sensors and
flag drift before failure. This is the core of a predictive-maintenance product.

In [ ]:
import torch
import numpy as np
import timesfm

torch.set_float32_matmul_precision("high")

# Downloads ~800 MB of weights the first time, then caches in ~/.cache/huggingface/
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)

model.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)
print("Model loaded and compiled.")

In [ ]:
# A vibration sensor that slowly drifts upward (bearing wearing out)
rng = np.random.default_rng(77)
t = np.arange(500)
healthy = 2.0 + 0.3*np.sin(2*np.pi*t/50) + rng.normal(0, 0.05, t.size)
drift = np.where(t > 400, 0.004*(t-400), 0.0)     # fault starts at t=400
signal = (healthy + drift).astype(np.float32)

## 1. Forecast the *expected* healthy behaviour from clean history

In [ ]:
train = signal[:400]                 # only healthy portion
H = 100
point, q = model.forecast(horizon=H, inputs=[train])
point, q = point[0], q[0]
future_actual = signal[400:500]      # what really happened (with the fault)

In [ ]:
IDX_Q10, IDX_Q90 = 1, 9
breaches = (future_actual < q[:, IDX_Q10]) | (future_actual > q[:, IDX_Q90])
first_alert = int(np.argmax(breaches)) if breaches.any() else None
print("first out-of-band step after fault onset:",
      first_alert, "(alert fires early enough to schedule maintenance)")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
xf = range(400, 500)
fig, ax = plt.subplots(figsize=(14,5))
ax.plot(range(500), signal, color="tab:blue", lw=1, label="sensor")
ax.plot(xf, point, color="tab:orange", lw=1.5, label="expected (healthy)")
ax.fill_between(xf, q[:,IDX_Q10], q[:,IDX_Q90], color="tab:orange", alpha=0.2, label="normal band")
ax.scatter([400+i for i in range(100) if breaches[i]],
           [future_actual[i] for i in range(100) if breaches[i]],
           color="red", s=15, zorder=5, label="alert")
ax.axvline(400, ls="--", color="grey"); ax.set_title("Predictive maintenance alert")
ax.legend(); fig.tight_layout(); fig.savefig("iot_maintenance.png", dpi=130)
print("saved iot_maintenance.png")

### Turn it into a service
- Ingest sensor streams → periodically forecast the healthy envelope.
- Fire a Slack/email alert the moment readings leave the band.
- Sell per-device/month monitoring with a maintenance dashboard.